# Binary Exoplanet Prediction Project

Clean workflow: minimal descriptive exploration using G01 tools, then binary supervised prediction of `CONFIRMED` vs `FALSE POSITIVE` with a Decision Tree baseline and XGBoost model.

In [ ]:
# --- 1. Imports and Dataset Loading ---
# G01 tools used here: imports, variables, print, pandas dataframe inspection.
# The notebook is designed for a Colab-like environment, matching the original project setup.

import os
import sys
import subprocess

try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'kagglehub'])
    import kagglehub

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

pd.options.display.float_format = '{:.2f}'.format

# Download and load the NASA Kepler cumulative dataset.
path = kagglehub.dataset_download('nasa/kepler-exoplanet-search-results')
full_path = os.path.join(path, 'cumulative.csv')
df = pd.read_csv(full_path)

print('Dataset loaded successfully.')
print(f'Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
display(df.head())

In [ ]:
# --- 2. First Data Inspection with G01 Tools ---
# G01 instruments: shape, columns, info, head.
# Purpose: understand dimensions, variable names, types, and missingness before any modeling.

print('Shape:')
print(df.shape)

print('\nColumn names:')
print(df.columns.tolist())

print('\nDataFrame information:')
df.info()

print('\nFirst rows:')
display(df.head())

In [ ]:
# --- 3. Dataset Dictionary Guided Cleaning ---
# This is the single cleaning step used by the whole notebook.
# Metadata and leakage columns are removed from the predictive dataframe.
# False-positive diagnostic flags are preserved separately for descriptive analysis only.

RANDOM_STATE = 42
TARGET_COLUMN = 'koi_disposition'
POSITIVE_LABEL = 'CONFIRMED'
NEGATIVE_LABEL = 'FALSE POSITIVE'
CANDIDATE_LABEL = 'CANDIDATE'

# Diagnostic false-positive flags from the data dictionary.
# They are post-analysis conclusions, so using them as features would leak the answer.
fp_flag_cols = [
    'koi_fpflag_nt',
    'koi_fpflag_ss',
    'koi_fpflag_co',
    'koi_fpflag_ec'
]

# Metadata columns: useful for cataloging, not for learning physical patterns.
metadata_cols = [
    'rowid',
    'kepid',
    'kepoi_name',
    'koi_tce_delivname',
    'koi_tce_plnt_num'
]

# Leakage columns: they encode naming, preliminary classification, or NASA probability scores.
leakage_cols = [
    'kepler_name',
    'koi_pdisposition',
    'koi_score'
]

# Structurally empty columns found in the missing-value audit.
empty_cols = ['koi_teq_err1', 'koi_teq_err2']

columns_to_drop = metadata_cols + leakage_cols + fp_flag_cols

# Keep flags separately before dropping them from the cleaned predictive dataframe.
df_false_positive_flags = df.loc[
    df[TARGET_COLUMN].eq(NEGATIVE_LABEL),
    [TARGET_COLUMN, *fp_flag_cols]
].copy()

# df_clean is the unified dataframe for EDA and modeling.
# It still contains koi_disposition because the target is needed later.
df_clean = df.drop(columns=[c for c in columns_to_drop if c in df.columns])

print(f'Columns before cleaning: {df.shape[1]}')
print(f'Columns after metadata/leakage removal: {df_clean.shape[1]}')
print(f'False-positive flag rows preserved: {df_false_positive_flags.shape[0]}')
print('Dropped columns:')
print(columns_to_drop)

In [ ]:
# --- 4. Target Distribution with G01 Categorical Tools ---
# G01 instruments: value_counts, normalize=True, countplot.
# Purpose: inspect the target before choosing the supervised-learning task.

print('Target counts:')
display(df_clean[TARGET_COLUMN].value_counts(dropna=False).to_frame('count'))

print('Target proportions:')
display(df_clean[TARGET_COLUMN].value_counts(normalize=True, dropna=False).to_frame('proportion'))

plt.figure(figsize=(7, 4))
sns.countplot(data=df_clean, x=TARGET_COLUMN, hue=TARGET_COLUMN, palette='Set1', legend=False)
plt.title('KOI Disposition Counts')
plt.xlabel('Disposition')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print('Project decision: CANDIDATE observations are excluded from supervised training because they are unresolved labels.')

In [ ]:
# --- 5. Missing Values Audit with G01 Tools ---
# G01 instruments: isnull, sum, percentages, sorting, display.
# Purpose: identify what must be handled later by train-only imputation.

missing_counts = df_clean.isnull().sum()
missing_percentage = (missing_counts / len(df_clean)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_counts,
    'Percentage (%)': missing_percentage
})

missing_df = missing_df[missing_df['Missing Values'] > 0].sort_values('Missing Values', ascending=False)

print('Missing-value audit after metadata/leakage removal:')
display(missing_df)

print('Important modeling rule: missing values are not imputed here. Imputation will be fitted inside Pipelines after train/test split.')

In [ ]:
# --- 6. Numerical Descriptive Statistics with G01 Tools ---
# G01 instruments: describe, mean, median, std, quantile/IQR.
# Purpose: summarize the main physical variables without advanced preprocessing.

key_num_vars = [
    'koi_period',
    'koi_duration',
    'koi_depth',
    'koi_prad',
    'koi_teq',
    'koi_insol',
    'koi_model_snr',
    'koi_steff',
    'koi_slogg',
    'koi_srad',
    'koi_kepmag'
]
key_num_vars = [c for c in key_num_vars if c in df_clean.columns]

print('Descriptive statistics for key physical variables:')
display(df_clean[key_num_vars].describe())

summary_rows = []
for col in key_num_vars:
    x = df_clean[col]
    summary_rows.append({
        'feature': col,
        'mean': x.mean(),
        'median': x.median(),
        'std': x.std(),
        'iqr': x.quantile(0.75) - x.quantile(0.25),
        'min': x.min(),
        'max': x.max()
    })

summary_table = pd.DataFrame(summary_rows)
display(summary_table)

In [ ]:
# --- 7. Necessary Visual EDA with G01 Plots ---
# G01 instruments: histplot, boxplot, scatterplot, correlation heatmap.
# Purpose: inspect distribution shape and class separation using simple descriptive plots.

# Histograms: check skewness/shape for physically important variables.
# Speed choice: no KDE, because KDE can be slow on skewed variables.
# Log scale choice: use logarithmic bins, not only a log x-axis, so the bars are meaningful.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, feature in zip(axes, ['koi_prad', 'koi_period', 'koi_depth']):
    if feature in df_clean.columns:
        plot_data = df_clean[df_clean[feature] > 0]
        log_bins = np.logspace(np.log10(plot_data[feature].min()), np.log10(plot_data[feature].max()), 30)
        sns.histplot(data=plot_data, x=feature, bins=log_bins, alpha=0.4, edgecolor='white', ax=ax)
        ax.set_xscale('log')
        ax.set_title(f'Distribution of {feature} (log bins)')
plt.tight_layout()
plt.show()

# Boxplots: compare key variables across the target categories.
# The y-axis is log-scaled to make class differences visible despite extreme values.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, feature in zip(axes, ['koi_prad', 'koi_period', 'koi_model_snr']):
    if feature in df_clean.columns:
        plot_data = df_clean[df_clean[feature] > 0]
        sns.boxplot(data=plot_data, x=TARGET_COLUMN, y=feature, hue=TARGET_COLUMN, palette='Set1', legend=False, ax=ax)
        ax.set_yscale('log')
        ax.set_title(f'{feature} by disposition (log scale)')
        ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

# Scatterplot: inspect the relationship between orbital period and planetary radius.
# Speed/readability choice: use a reproducible sample if the dataset is large.
# Both axes are log-scaled because period and radius cover very different orders of magnitude.
if {'koi_period', 'koi_prad'}.issubset(df_clean.columns):
    scatter_data = df_clean[(df_clean['koi_period'] > 0) & (df_clean['koi_prad'] > 0)]
    scatter_data = scatter_data.sample(n=min(2000, len(scatter_data)), random_state=RANDOM_STATE)
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=scatter_data, x='koi_period', y='koi_prad', hue=TARGET_COLUMN, alpha=0.6, palette='Set1', s=25)
    plt.xscale('log')
    plt.yscale('log')
    plt.title('Period vs Planet Radius by Disposition (log-log scale)')
    plt.tight_layout()
    plt.show()

# Correlation: simple linear relationships among selected numerical variables.
corr_vars = [c for c in ['koi_period', 'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_model_snr', 'koi_steff', 'koi_srad'] if c in df_clean.columns]
plt.figure(figsize=(9, 6))
sns.heatmap(df_clean[corr_vars].corr(numeric_only=True), annot=False, cmap='Blues', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Key Numerical Variables')
plt.tight_layout()
plt.show()

print('EDA conclusion: false positives show visible differences in physical/transit variables, while CANDIDATE is not a final supervised label.')

In [ ]:
# --- 8. Binary Modeling Dataset and Correct Split Order ---
# This cell begins the supervised-learning workflow.
# The order is: remove CANDIDATE -> define X/y -> train/test split -> impute inside Pipelines.

from sklearn.model_selection import train_test_split

# Keep only final labels for the binary supervised task.
df_binary = df_clean[df_clean[TARGET_COLUMN].isin([POSITIVE_LABEL, NEGATIVE_LABEL])].copy()

# Binary target: confirmed exoplanet = 1, false positive = 0.
y = df_binary[TARGET_COLUMN].map({NEGATIVE_LABEL: 0, POSITIVE_LABEL: 1})

# Build X from the cleaned dataframe.
# No imputation, scaling, fitting, or model training has happened before this split.
X = df_binary.drop(columns=[TARGET_COLUMN, *[c for c in empty_cols if c in df_binary.columns]])
X = X.select_dtypes(include=['float64', 'int64'])

# Safeguards against leakage and target mistakes.
assert set(df_binary[TARGET_COLUMN].unique()) == {POSITIVE_LABEL, NEGATIVE_LABEL}
assert CANDIDATE_LABEL not in set(df_binary[TARGET_COLUMN].unique())
assert y.notna().all()
assert not any(c in X.columns for c in columns_to_drop)
assert not any(c in X.columns for c in empty_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Binary modeling dataset prepared.')
print(f'Rows after removing CANDIDATE: {df_binary.shape[0]}')
print(f'Predictive features: {X.shape[1]}')
print(f'Training set: {X_train.shape[0]} rows')
print(f'Test set: {X_test.shape[0]} rows')
print('\nTarget distribution:')
print(y.map({0: NEGATIVE_LABEL, 1: POSITIVE_LABEL}).value_counts())
print('\nExcluded columns still in X:')
print([c for c in columns_to_drop + empty_cols if c in X.columns])

In [ ]:
# --- 9. Pipelines and Hyperparameter Grids ---
# Decision Tree is the interpretable baseline; XGBoost is the high-performance model.
# Median imputation is inside each Pipeline, so it is fitted only on training folds.
# StandardScaler is not used because both models are tree-based.

try:
    from xgboost import XGBClassifier
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'xgboost'])
    from xgboost import XGBClassifier

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline_tree_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

xgb_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('model', XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbosity=0
    ))
])

# Tree tuning: controls interpretability and overfitting.
tree_param_grid = {
    'model__max_depth': [2, 3, 4, 5],
    'model__min_samples_leaf': [10, 25, 50],
    'model__ccp_alpha': [0.0, 0.001, 0.005, 0.01]
}

# XGBoost tuning: compact grid for predictive power without excessive complexity.
xgb_param_grid = {
    'model__n_estimators': [50, 100, 150],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [2, 3],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}

print('Pipelines and grids ready.')

In [ ]:
# --- 10. Cross-Validation Model Selection ---
# GridSearchCV uses training data only.
# ROC-AUC is used because it evaluates probability ranking across thresholds.

tree_search = GridSearchCV(
    estimator=baseline_tree_pipeline,
    param_grid=tree_param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    refit=True
)

tree_search.fit(X_train, y_train)

xgb_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    refit=True
)

xgb_search.fit(X_train, y_train)

cv_results = pd.DataFrame({
    'Model': ['Decision Tree baseline', 'XGBoost'],
    'Best CV ROC-AUC': [tree_search.best_score_, xgb_search.best_score_],
    'Best Parameters': [tree_search.best_params_, xgb_search.best_params_]
})

print('Cross-validation model selection completed on training data only.')
display(cv_results)

In [ ]:
# --- 11. Final Test Evaluation ---
# The held-out test set is evaluated once after CV model selection.
# Hard labels support confusion matrices; probabilities support ROC-AUC.

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay
)

models = {
    'Decision Tree baseline': tree_search.best_estimator_,
    'XGBoost': xgb_search.best_estimator_
}

test_summary = []

for model_name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    test_summary.append({
        'Model': model_name,
        'Test ROC-AUC': roc_auc_score(y_test, y_prob),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision CONFIRMED': precision_score(y_test, y_pred, pos_label=1),
        'Recall CONFIRMED': recall_score(y_test, y_pred, pos_label=1),
        'F1 CONFIRMED': f1_score(y_test, y_pred, pos_label=1)
    })

    print(f'\n=== {model_name} ===')
    print(classification_report(
        y_test,
        y_pred,
        target_names=[NEGATIVE_LABEL, POSITIVE_LABEL],
        digits=4
    ))

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        display_labels=[NEGATIVE_LABEL, POSITIVE_LABEL],
        cmap='Blues',
        values_format='d'
    )
    plt.title(f'Confusion Matrix - {model_name}')
    plt.show()

test_summary = pd.DataFrame(test_summary).sort_values('Test ROC-AUC', ascending=False)
display(test_summary)

print('Error interpretation:')
print('- FALSE POSITIVE predicted as CONFIRMED = false scientific confirmation.')
print('- CONFIRMED predicted as FALSE POSITIVE = missed exoplanet.')

In [ ]:
# --- 12. Overfitting Check ---
# Purpose: compare training, cross-validation, and test performance.
# If training ROC-AUC is much higher than CV/test ROC-AUC, the model may be overfitting.
# The 0.05 gap used below is a practical warning threshold, not a formal statistical test.

overfitting_rows = []

for model_name, search in [('Decision Tree baseline', tree_search), ('XGBoost', xgb_search)]:
    model = search.best_estimator_

    train_prob = model.predict_proba(X_train)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]

    train_auc = roc_auc_score(y_train, train_prob)
    cv_auc = search.best_score_
    test_auc = roc_auc_score(y_test, test_prob)

    train_cv_gap = train_auc - cv_auc
    train_test_gap = train_auc - test_auc

    if train_cv_gap > 0.05 or train_test_gap > 0.05:
        conclusion = 'Possible overfitting signal'
    else:
        conclusion = 'No strong overfitting signal'

    overfitting_rows.append({
        'Model': model_name,
        'Train ROC-AUC': train_auc,
        'Best CV ROC-AUC': cv_auc,
        'Test ROC-AUC': test_auc,
        'Train-CV gap': train_cv_gap,
        'Train-Test gap': train_test_gap,
        'Interpretation': conclusion
    })

overfitting_check = pd.DataFrame(overfitting_rows)
display(overfitting_check)

overfitting_plot = overfitting_check.melt(
    id_vars='Model',
    value_vars=['Train ROC-AUC', 'Best CV ROC-AUC', 'Test ROC-AUC'],
    var_name='Evaluation set',
    value_name='ROC-AUC'
)

plt.figure(figsize=(9, 5))
sns.barplot(data=overfitting_plot, x='Model', y='ROC-AUC', hue='Evaluation set')
plt.ylim(0.5, 1.0)
plt.title('Overfitting Check: Train vs CV vs Test ROC-AUC')
plt.tight_layout()
plt.show()

print('Reading rule: a large Train-CV or Train-Test gap suggests overfitting.')
print('A small gap between CV and test ROC-AUC suggests that model selection generalized reasonably well.')

In [ ]:
# --- 13. Decision Tree Visualization ---
# G07 instrument: plot_tree.
# Purpose: make the baseline model interpretable by showing its split rules.
# This plot explains the selected Decision Tree baseline, not the XGBoost model.

from sklearn.tree import plot_tree

best_tree_pipeline = tree_search.best_estimator_
best_tree_model = best_tree_pipeline.named_steps['model']

plt.figure(figsize=(22, 10))
plot_tree(
    best_tree_model,
    feature_names=X.columns,
    class_names=[NEGATIVE_LABEL, POSITIVE_LABEL],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8
)
plt.title('Decision Tree Baseline - First Levels of the Tree')
plt.show()

print('Interpretation note: start from the root node and follow the yes/no split rules until a leaf node.')
print('Only the first levels are shown to keep the plot readable; the tuned tree may contain additional deeper splits.')

In [ ]:
# --- 14. XGBoost Feature Importance ---
# Feature importance connects the final predictive model back to physical variables.
# It is model explanation, not causal evidence.

best_xgb_model = xgb_search.best_estimator_.named_steps['model']
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('Top XGBoost feature importances:')
display(feature_importance.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(
    data=feature_importance.head(15),
    x='Importance',
    y='Feature',
    color='#cc2b2b'
)
plt.title('Top 15 XGBoost Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# --- 15. Descriptive False-Positive Flag Extension ---
# The flags are not mutually exclusive, so they are not modeled as one multiclass target.
# They are analyzed descriptively to explain false-positive diagnosis overlap.

flag_display_names = {
    'koi_fpflag_nt': 'Not Transit-like',
    'koi_fpflag_ss': 'Stellar Eclipse',
    'koi_fpflag_co': 'Centroid Offset',
    'koi_fpflag_ec': 'Ephemeris Match'
}

fp_flags = df_false_positive_flags[fp_flag_cols].astype(int).copy()

flag_counts = fp_flags.sum().rename(index=flag_display_names).sort_values(ascending=False)
active_flag_counts = fp_flags.sum(axis=1).value_counts().sort_index()
flag_combinations = fp_flags.astype(str).agg(''.join, axis=1).value_counts().sort_index()
flag_cooccurrence = fp_flags.T.dot(fp_flags)
flag_cooccurrence.index = [flag_display_names[c] for c in flag_cooccurrence.index]
flag_cooccurrence.columns = [flag_display_names[c] for c in flag_cooccurrence.columns]

print('False-positive flag totals:')
display(flag_counts.to_frame('Count'))

print('Number of active flags per FALSE POSITIVE observation:')
display(active_flag_counts.to_frame('Number of observations'))

print('Flag overlap combinations, ordered as nt-ss-co-ec:')
display(flag_combinations.to_frame('Number of observations'))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(x=flag_counts.values, y=flag_counts.index, color='#cc2b2b', ax=axes[0])
axes[0].set_title('False-Positive Diagnostic Flag Totals')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Flag')

sns.barplot(x=active_flag_counts.index.astype(str), y=active_flag_counts.values, color='#baddf5', ax=axes[1])
axes[1].set_title('Active Flag Count per False Positive')
axes[1].set_xlabel('Number of active flags')
axes[1].set_ylabel('Number of observations')

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(flag_cooccurrence, annot=True, fmt='d', cmap='Reds')
plt.title('False-Positive Flag Co-occurrence')
plt.tight_layout()
plt.show()

multiple_flag_rows = int((fp_flags.sum(axis=1) > 1).sum())
zero_flag_rows = int((fp_flags.sum(axis=1) == 0).sum())
total_fp_rows = int(fp_flags.shape[0])

print(f'Total FALSE POSITIVE observations: {total_fp_rows}')
print(f'FALSE POSITIVE observations with more than one active flag: {multiple_flag_rows}')
print(f'FALSE POSITIVE observations with zero active flags: {zero_flag_rows}')
print('\nConclusion:')
print('The false-positive labels overlap, so a single multiclass model would be statistically inappropriate.')
print('The main supervised model remains CONFIRMED vs FALSE POSITIVE; the flag analysis is descriptive only.')

In [ ]:
# --- 16. Project Summary ---
# Final methodological summary for the notebook.

print('Summary:')
print('1. G01 descriptive tools were used for only the necessary data inspection and EDA.')
print('2. Metadata, leakage variables, and diagnostic flags were removed from predictive X.')
print('3. CANDIDATE observations were excluded from supervised training because they are unresolved.')
print('4. Train/test split was performed before imputation and model fitting.')
print('5. Decision Tree provides the interpretable baseline; XGBoost provides the stronger boosted-tree model.')
print('6. Overfitting was checked by comparing train, cross-validation, and test ROC-AUC.')
print('7. False-positive diagnostic flags were analyzed descriptively because they are not mutually exclusive.')